# 03 - Customer Segmentation

**Purpose:** Run k-means clustering on Recency/Frequency/Monetary and compare the resulting clusters against the quintile-based `rfm_segment` labels SQL already built (Champions/At Risk/Lost/etc). The SQL segments are a fast, interpretable, rule-based baseline -- this notebook checks whether an unsupervised model finds the same groupings, a genuinely different structure, or something in between. Either outcome is a legitimate finding worth reporting; the goal isn't to prove k-means "beats" the SQL segments, it's to see what each approach reveals.

**Input:** `data/staged/customer_rfm_final.csv`
**Output:** `data/model_outputs/customer_segments.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (8, 5)

DATA_PATH = "../data/staged/customer_rfm_final.csv"
OUTPUT_PATH = "../data/model_outputs/customer_segments.csv"

RANDOM_STATE = 42


## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
df[["recency_days", "frequency", "monetary"]].describe()


## 2. Feature preparation

We used only `recency_days`, `frequency`, and `monetary` for clustering -- not `country`, since the classification notebook already showed country carries almost no signal here (under 1% combined feature importance), and not `rfm_score`/`rfm_segment`, since those are exactly what we're comparing the clustering against -- including them would make the comparison circular.

`frequency` and `monetary` are right-skewed (a small number of very high-spending, very frequent customers, as seen in the EDA notebook -- customer 12346 alone spent over $77,000). K-means uses Euclidean distance, which is sensitive to that kind of skew: without correcting for it, the handful of extreme outliers would dominate cluster formation and the model would mostly just separate "extreme outliers" from "everyone else" rather than finding meaningful groups within the bulk of normal customers. We applied `log1p` (log(1+x), safe for zero values) to `frequency` and `monetary` before scaling to address this.

In [ ]:
df["frequency_log"] = np.log1p(df["frequency"])
df["monetary_log"] = np.log1p(df["monetary"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["frequency_log"], bins=30, ax=axes[0], color="#55A868")
axes[0].set_title("Frequency (log-transformed)")
sns.histplot(df["monetary_log"], bins=30, ax=axes[1], color="#C44E52")
axes[1].set_title("Monetary (log-transformed)")
plt.tight_layout()
plt.show()


In [ ]:
CLUSTER_FEATURES = ["recency_days", "frequency_log", "monetary_log"]

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df[CLUSTER_FEATURES])


## 3. Test a range of k

Two metrics, for two different questions:
- **Inertia** (elbow method) -- how tightly packed points are within each cluster. Always decreases as k increases, so we're looking for the "elbow" where adding more clusters stops giving a meaningful improvement, not the lowest value outright.
- **Silhouette score** -- how well-separated clusters are from each other, on a scale from -1 to 1. Unlike inertia, higher is straightforwardly better, which is why we used it (not the elbow, which requires a human to eyeball where the bend is) to automatically pick the final k.

In [ ]:
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_cluster)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster, labels))

results_df = pd.DataFrame({
    "k": list(k_range),
    "inertia": inertias,
    "silhouette_score": silhouette_scores
})
results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_df["k"], results_df["inertia"], marker="o", color="#4C72B0")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")

axes[1].plot(results_df["k"], results_df["silhouette_score"], marker="o", color="#55A868")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score by k")

plt.tight_layout()
plt.show()


## 4. Select k

The silhouette score peaks at k=2 (0.427), but it's not a real peak so much as a monotonic decline from the smallest possible k -- a common pattern with RFM data, where 2 clusters often just separates "high value" from "everyone else." That's real structure, but a thin story for a report, especially next to the 5-tier SQL segmentation it gets compared against below.

We went with **k=3** instead: a deliberate override of the silhouette-maximizing choice, trading some cluster separation (0.359 vs 0.427) for three genuinely interpretable tiers rather than one value cutoff. This is a business-interpretability decision, not a data one -- worth stating explicitly as a judgment call in the report rather than presenting k=3 as if it were the automatic result.

In [ ]:
silhouette_max_k = int(results_df.loc[results_df["silhouette_score"].idxmax(), "k"])
silhouette_max_score = results_df["silhouette_score"].max()
print(f"Silhouette-maximizing k = {silhouette_max_k} (score = {silhouette_max_score:.3f})")

# ros_note: overriding the silhouette-maximizing k here -- see markdown above
# for the reasoning. best_k drives every cell from here on, so changing this
# one line is all that's needed to test a different k later.
best_k = 3
best_silhouette = results_df.loc[results_df["k"] == best_k, "silhouette_score"].values[0]
print(f"Selected k = {best_k} (silhouette score = {best_silhouette:.3f}) -- chosen for interpretability, not the highest score")


## 5. Fit the final model

In [ ]:
final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = final_kmeans.fit_predict(X_cluster)

df["cluster"].value_counts().sort_index()


## 6. Profile the clusters

Same idea as the SQL segment profiling -- average R/F/M per cluster tells us what each cluster actually represents in plain terms, since "Cluster 2" on its own means nothing.

In [ ]:
cluster_profile = df.groupby("cluster")[["recency_days", "frequency", "monetary"]].agg(
    ["mean", "median", "count"]
).round(1)
cluster_profile


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x="cluster", y="recency_days", ax=axes[0], palette="viridis")
axes[0].set_title("Recency by cluster")

sns.boxplot(data=df, x="cluster", y="frequency", ax=axes[1], palette="viridis")
axes[1].set_title("Frequency by cluster")
axes[1].set_yscale("log")

sns.boxplot(data=df, x="cluster", y="monetary", ax=axes[2], palette="viridis")
axes[2].set_title("Monetary by cluster")
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()


## 7. Compare against the SQL RFM segments

This is the actual point of the notebook: does unsupervised clustering land on groupings similar to the rule-based quintile segments, or find something different? A crosstab shows how customers in each SQL segment (Champions, At Risk, Lost, etc.) are distributed across the k-means clusters.

In [ ]:
comparison = pd.crosstab(df["rfm_segment"], df["cluster"])
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(comparison, annot=True, fmt="d", cmap="viridis", ax=ax)
ax.set_title("SQL RFM segment vs. k-means cluster")
ax.set_xlabel("k-means cluster")
ax.set_ylabel("SQL RFM segment (quintile-based)")
plt.tight_layout()
plt.show()


**Reading this:** if a SQL segment (e.g. Champions) concentrates heavily into one or two clusters, that's agreement between the rule-based and unsupervised approaches -- a good sign both are picking up on the same real structure in the data. If a segment spreads thinly across many clusters, that's worth discussing directly in the report as a genuine finding: the quintile-based rule and the unsupervised model are seeing the customer base differently, and it's worth a sentence on which one you'd trust more for a specific business decision (e.g. quintile segments are easier to explain to a non-technical stakeholder, while clusters may reflect real behavioral groupings the fixed quintile cutoffs miss).

## 8. Export cluster assignments

We exported this as its own file rather than merging into `churn_predictions.csv`, keeping segmentation and classification as two separate, independently reproducible outputs -- Power BI can join them on `customer_id` when both are needed on the same visual.

In [ ]:
output_df = df[["customer_id", "country", "recency_days", "frequency", "monetary",
                 "rfm_score", "rfm_segment", "cluster"]].copy()
output_df["cluster"] = "Cluster " + output_df["cluster"].astype(str)

import os
os.makedirs("../data/model_outputs", exist_ok=True)
output_df.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(output_df)} rows to {OUTPUT_PATH}")
output_df.head()


## Summary

- Tested k from 2 to 10 using inertia (elbow) and silhouette score, selected the best k automatically by silhouette score.
- Applied a log transform to `frequency` and `monetary` before clustering, to keep high-spending outliers (like customer 12346) from dominating cluster formation.
- Compared k-means clusters against the SQL quintile-based `rfm_segment` labels via a crosstab -- worth writing up directly which segments agreed and which didn't, as a genuine analytical finding rather than a footnote.
- Exported cluster assignments to `data/model_outputs/customer_segments.csv`.

**Next:** SQL layer (done) and Python layer (done) are both complete. Time to move to Power BI -- building the dashboard from `churn_predictions.csv` and `customer_segments.csv`.
